<a href="https://colab.research.google.com/github/mentlana/Analysis-of-symbols-and-standards-in-Muse/blob/main/%D0%AD%D1%82%D0%B0%D0%BF_1_%D0%A4%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%BA%D0%BE%D1%80%D0%BF%D1%83%D1%81%D0%B0_%D0%B8_%D0%BF%D1%80%D0%B5%D0%B4%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%AD%D1%82%D0%B0%D0%BF_1_%D0%A4%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%BA%D0%BE%D1%80%D0%BF%D1%83%D1%81%D0%B0_%D0%B8_%D0%BF%D1%80%D0%B5%D0%B4%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 1. Формирование корпуса и предобработка

Импорт необходимых библиотек

In [ ]:
import pandas as pd
from collections import Counter
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import stopwords
import string
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Удаляем пустые строки, то есть строки, где нет текстового содержимого в колонке lyrics_raw

In [ ]:
df = pd.read_csv("muse_corpus.csv")
df = df[df['lyrics_raw'].notna()]
df = df[df['lyrics_raw'].str.strip() != ""]
df = df.reset_index(drop=True)

In [ ]:
print(f"Количество строк после удаления: {len(df)}")
total_words = df['lyrics_raw'].str.split().str.len().sum()
print(f"Всего слов в колонке lyrics_raw: {total_words}")

Количество строк после удаления: 106
Всего слов в колонке lyrics_raw: 18620


Очистка и предобработка текстов: удаление характерной для Genius.com разметки текста [Verse], [Chorus], лишних переносов, изолированных цифр, пунктуации и приведение текста к нижнему регистру.

In [23]:
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    contractions = {
        "I'll": "I will",
        "You'll": "You will",
        "He'll": "He will",
        "She'll": "She will",
        "It'll": "It will",
        "We'll": "We will",
        "They'll": "They will",
        "I'm": "I am",
        "You're": "You are",
        "He's": "He is",
        "She's": "She is",
        "It's": "It is",
        "We're": "We are",
        "They're": "They are",
        "Don't": "Do not",
        "Doesn't": "Does not",
        "Didn't": "Did not",
        "Can't": "Cannot",
        "Won't": "Will not",
        "Wouldn't": "Would not",
        "Couldn't": "Could not",
        "Shouldn't": "Should not",
        "I've": "I have",
        "You've": "You have",
        "We've": "We have",
        "They've": "They have",
        "I'd": "I would",
        "You'd": "You would",
        "He'd": "He would",
        "She'd": "She would",
        "It'd": "It would",
        "We'd": "We would",
        "They'd": "They would"
    }

    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
        text = text.replace(contraction.lower(), expansion.lower())
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    return text

df['lyrics_clean'] = df['lyrics_raw'].apply(clean_lyrics)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         106 non-null    object
 1   album         106 non-null    object
 2   year          106 non-null    int64 
 3   lyrics_raw    106 non-null    object
 4   lyrics_clean  106 non-null    object
 5   tokens        106 non-null    object
 6   tokens_str    106 non-null    object
 7   lemma_count   106 non-null    int64 
dtypes: int64(2), object(6)
memory usage: 6.8+ KB


In [25]:
print(df.shape)
df.head(5)

(106, 8)


,title,album,year,lyrics_raw,lyrics_clean,tokens,tokens_str,lemma_count
0,Sunburn,Showbiz,1999,[Verse 1]\nCome waste your millions here\nSecr...,\ncome waste your millions here\nsecretly she ...,"[come, waste, million, secretly, sneer, anothe...",come waste million secretly sneer another corp...,79
1,Muscle Museum,Showbiz,1999,[Verse 1]\nShe had something to confess to\nBu...,\nshe had something to confess to\nbut you do ...,"[something, confess, dont, time, look, way, wa...",something confess dont time look way wait reve...,75
2,Fillip,Showbiz,1999,"[Verse 1]\nIt's happening soon, it's happening...",\nit is happening soon it is happening soon\ni...,"[happening, soon, happening, soon, scent, blow...",happening soon happening soon scent blowing di...,48
3,Falling Down,Showbiz,1999,[Verse 1]\nI'm falling down\nAnd fifteen thous...,\ni am falling down\nand fifteen thousand peop...,"[im, falling, fifteen, thousand, people, screa...",im falling fifteen thousand people scream begg...,77
4,Cave,Showbiz,1999,"[Verse 1]\nLeave me alone, it's nothing seriou...",\nleave me alone it is nothing serious\ni will...,"[leave, alone, nothing, serious, ill, got, nou...",leave alone nothing serious ill got nought the...,68


Токенизация (+удаление стоп слов) и лемматизация текстов

In [26]:
extra_stopwords = {'woah', 'hey', 'ooh', 'yeah', 'ah', 'oh', 'la', 'na'}
stop_words = set(stopwords.words('english')).union(extra_stopwords)

lemmatizer = WordNetLemmatizer()
def tokenize_and_lemmatize(text):
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmas

df['tokens_str'] = df['tokens'].apply(' '.join)
df['lemma_count'] = df['tokens'].apply(len)

In [28]:
df['tokens'] = df['lyrics_clean'].apply(tokenize_and_lemmatize)
df['tokens_str'] = df['tokens'].apply(' '.join)
df['lemma_count'] = df['tokens'].apply(len)

df.to_csv('muse_corpus_full.csv', index=False, encoding='utf-8')
print("Сохранён: muse_corpus_full.csv")

with open('muse_corpus_for_antconc.txt', 'w', encoding='utf-8') as f:
    for lyrics in df['lyrics_clean']:
        f.write(lyrics + '\n\n')
print("Сохранён: muse_corpus_for_antconc.txt")

Сохранён: muse_corpus_full.csv
Сохранён: muse_corpus_for_antconc.txt


В результате на выходе два файла:
*   muse_corpus_full.csv – полная таблица с метаданными, очищенными текстами и токенами.
*   muse_corpus_for_antconc.txt – для анализа в корпусных менеджерах (AntConc).

In [29]:
print("\nСТАТИСТИКА КОРПУСА")
print(f"Всего песен: {len(df)}")
print(f"Средняя длина текста песни: {df['lemma_count'].mean():.1f}")
print(f"Всего слов: {df['lemma_count'].sum()}")
print(f"Общее количество уникальных лемм: {len(set([lemma for sublist in df['tokens'] for lemma in sublist]))}")


СТАТИСТИКА КОРПУСА
Всего песен: 106
Средняя длина текста песни: 75.0
Всего слов: 7951
Общее количество уникальных лемм: 1765
